In [7]:
# 检查数据
import os
import shutil

# 数据集存放目录
DATA_DIR = "./data"
TRAIN_PATH = os.path.join(DATA_DIR, "train.csv")
TEST_PATH = os.path.join(DATA_DIR, "test.csv")

# 数据集下载地址
KAGGLE_URL = "https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data"

def prepare_data():
    # 创建文件夹
    if not os.path.exists(DATA_DIR):
        os.makedirs(DATA_DIR)
        print(f"✅ 已创建文件夹：{DATA_DIR}")

    # 检查文件
    has_train = os.path.exists(TRAIN_PATH)
    has_test = os.path.exists(TEST_PATH)

    if has_train and has_test:
        print("✅ 数据集已存在，可以直接开始训练！")
        return True
    else:
        print("⚠️ 缺少数据集文件！")
        print(f"请前往 Kaggle 下载：{KAGGLE_URL}")
        print("下载后把 train.csv、test.csv 放到 ./data/ 文件夹下面")
        print("放好之后，重新运行这个单元格即可")
        return False

# 执行
prepare_data()

# 导入工具
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import cross_val_score

# 1.读取数据集
data = pd.read_csv('./data/train.csv', keep_default_na=False, na_values=[''])
test = pd.read_csv('./data/test.csv', keep_default_na=False, na_values=[''])
test_ids = test['Id'].copy()

# 2.观察数据结构
print(data.head(2))
print(data.isnull().sum())
print(data['SalePrice'].describe())

# 3.特征工程预处理
def feature_engineering(df):
    df = df.copy()  #复制独立数据避免污染原始数据

    # 处理‘LotFrontage’259 个缺失值，填充中位数
    df['LotFrontage'] = pd.to_numeric(df['LotFrontage'], errors='coerce')
    df['LotFrontage_缺失'] = df['LotFrontage'].isnull().astype(int)
    df['LotFrontage'] = df['LotFrontage'].fillna(df['LotFrontage'].median())

    # 处理‘MasVnrType’（填充众数），‘MasVnrArea’（填充中位数）缺失值
    df['MasVnrType'] = df['MasVnrType'].fillna(df['MasVnrType'].mode()[0])
    
    df['MasVnrArea'] = pd.to_numeric(df['MasVnrArea'], errors='coerce')
    df['MasVnrArea'] = df['MasVnrArea'].fillna(df['MasVnrArea'].median())
    
    # 处理有序编码
    Functional_map = {'Typ':8, 'Min1':7, 'Min2':6, 'Mod':5, 'Maj1':4, 'Maj2':3, 'Sev':2, 'Sal':1, 'NA':0}
    quality_map_6na = {'GLQ': 6, 'ALQ':5, 'BLQ':4, 'Rec':3, 'LwQ':2, 'Unf':1, 'NA':0}
    quality_map_5 = {'Ex': 5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1}
    quality_map_5na = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'NA': 0}
    quality_map_4na = {'Gd':4, 'Av':3, 'Mn':2, 'No':1, 'NA':0}
    GarageFinish_map = {'Fin':3, 'RFn':2, 'Unf':1, 'NA':0}
    
    df['ExterQual_编码'] = df['ExterQual'].map(quality_map_5)  #评估外部材料的质量
    df['ExterCond_编码'] = df['ExterCond'].map(quality_map_5)  #评估材料外部的当前状况
    df['BsmtQual_编码'] =df['BsmtQual'].map(quality_map_5na)  #评估地下室的高度
    df['BsmtCond_编码'] = df['BsmtCond'].map(quality_map_5na)  #评估地下室的整体状况
    df['BsmtExposure_编码'] = df['BsmtExposure'].map(quality_map_4na)  #地下室暴露面
    df['BsmtFinType1_编码'] = df['BsmtFinType1'].map(quality_map_6na)  #地下室完工面积评级
    df['BsmtFinType2_编码'] = df['BsmtFinType2'].map(quality_map_6na)  #地下室装修面积评级
    df['HeatingQC_编码'] = df['HeatingQC'].map(quality_map_5)  #供暖质量与状况
    df['KitchenQual_编码'] = df['KitchenQual'].map(quality_map_5na)  #厨房质量
    df['Functional_编码'] = df['Functional'].map(Functional_map)  #家庭功能
    df['FireplaceQu_编码'] = df['FireplaceQu'].map(quality_map_5na)  #壁炉质量
    df['GarageType_编码'] = df['GarageType'].map({'2Types':6, 'Attchd':5, 'Basment':4, 'BuiltIn':3, 'CarPort':2, 'Detchd':1, 'NA':0})  #车库类型
    df['GarageFinish_编码'] = df['GarageFinish'].map(GarageFinish_map)  #车库装修
    df['GarageQual_编码'] = df['GarageQual'].map(quality_map_5na)  #车库质量
    df['GarageCond_编码'] = df['GarageCond'].map(quality_map_5na)  #车库状况
    df['PoolQC_编码'] = df['PoolQC'].map({'Ex': 4, 'Gd':3, 'TA':2, 'Fa':1, 'NA':0})

    # 检查有序编码结果
    print(df[['ExterQual_编码', 'ExterCond_编码', 'BsmtQual_编码', 'BsmtCond_编码', 'BsmtExposure_编码', 'BsmtFinType1_编码', 'BsmtFinType2_编码', 'HeatingQC_编码', 'KitchenQual_编码', 'Functional_编码', 'FireplaceQu_编码', 'GarageType_编码', 'GarageFinish_编码', 'GarageQual_编码', 'GarageCond_编码', 'PoolQC_编码']].isnull().sum())
    
    # 构建房龄特征
    df['房龄'] = df['YrSold'] - df['YearBuilt']
    df['改建房龄'] = df['YrSold'] - df['YearRemodAdd']

    # 构建车库特征
    df['GarageYrBlt'] = pd.to_numeric(df['GarageYrBlt'], errors='coerce')
    df['有车库'] = df['GarageYrBlt'].notnull().astype(int)
    df['车库房龄'] = np.where(
        df['GarageYrBlt'].notnull(),
        df['YrSold']-df['GarageYrBlt'],
        -1)

    # 转换'MSSubClass'格式
    df['MSSubClass'] = df['MSSubClass'].astype(str)

    # 对分类数据构建独热编码
    categorical_cols = ['MSSubClass', 'MSZoning', 'Alley', 'LotShape', 'LandContour', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'MasVnrType', 'Foundation', 'Heating', 'CentralAir', 'Electrical', 'PavedDrive', 'Fence', 'SaleType', 'SaleCondition','YrSold']
    cols_to_encode = [c for c in categorical_cols if c in df.columns]
    df = pd.get_dummies(df, columns=cols_to_encode)

   # 对MiscFeature杂项列独热
    df = pd.get_dummies(df, columns=['MiscFeature'], dummy_na=False)

    # 删除原始文本和无效列
    drop_cols = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
                 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual',
                 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
                 'GarageCond', 'PoolQC',
                 'Street', 'Utilities', 'Condition2', 'Exterior2nd',
                 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'Id']
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    
    # 转换所有列格式为数值（保底）
    df = df.apply(pd.to_numeric, errors='coerce')
    return df

# train 和 test 数据做特征工程
data_clean = feature_engineering(data)
test_clean = feature_engineering(test)

# 设定特征和标签
X = data_clean.drop(columns=['SalePrice'])
y = data_clean['SalePrice']

# 强制对齐测试集与训练集的特征列‌
test_clean = test_clean.reindex(columns=X.columns, fill_value=0)

# 特征裁剪：实验阶段选出的 80 列
final_features = pd.read_csv('top_features.txt', header=None)[0].tolist()
X_final = X[final_features]
test_final = test_clean[final_features]

# 训练前：对 y 做 log1p
y_log = np.log1p(y)


# 找出所有非法类型的列
invalid_cols = []
for col, dtype in X_final.dtypes.items():
    if dtype.name not in ['int64', 'float64', 'bool', 'category', 'int32', 'float32']:
        invalid_cols.append( (col, dtype) )

print("非法类型的列：", invalid_cols)

# 导入模型
xgb_model = xgb.XGBRegressor(
    n_estimators=452,         # 估计量
    learning_rate=0.05,       # 学习率
    max_depth=3,              # 学习深度
    min_child_weight=5,       # 叶节点最小样本权重
    reg_alpha=0.5,            # L1 正则化
    reg_lambda=2.0,           # L2 正则化
    subsample=0.7,            # 行采样比例
    colsample_bytree=0.6,     # 列采样比例
    random_state=42
)

# 模型训练
xgb_model.fit(X_final, y_log, verbose=False)

# 在全量上跑 5 折 CV，报告最终性能
scores = cross_val_score(
    xgb.XGBRegressor(
        n_estimators=452, learning_rate=0.05, max_depth=3,
        min_child_weight=5, reg_alpha=0.5, reg_lambda=2.0,
        subsample=0.7, colsample_bytree=0.6, random_state=42
    ),
    X_final, y_log, cv=5, scoring='r2'
)
print(f'最终模型 5 折 CV R²: {scores.mean():.4f} ± {scores.std():.4f}')

# 对 test.csv 预测，生成 submission
pred_log = xgb_model.predict(test_final)
pred = np.expm1(pred_log)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': pred})
submission.to_csv('submission.csv', index=False)
print(submission.head())
print(f'共生成 {len(submission)} 条预测')

import joblib
joblib.dump(xgb_model, 'house_price_model.joblib')
# 把 final_features（特征列名列表）也存下来！ FastAPI 接收数据时，必须按照这个顺序把数据喂给模型
import json
with open('features.json', 'w')as f:
    json.dump(final_features, f)
    
print("模型和特征列表已保存！")

✅ 数据集已存在，可以直接开始训练！
   Id  MSSubClass MSZoning LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL          65     8450   Pave    NA      Reg   
1   2          20       RL          80     9600   Pave    NA      Reg   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0     NA    NA          NA       0      2   
1         Lvl    AllPub  ...        0     NA    NA          NA       0      5   

  YrSold  SaleType  SaleCondition  SalePrice  
0   2008        WD         Normal     208500  
1   2007        WD         Normal     181500  

[2 rows x 81 columns]
Id               0
MSSubClass       0
MSZoning         0
LotFrontage      0
LotArea          0
                ..
MoSold           0
YrSold           0
SaleType         0
SaleCondition    0
SalePrice        0
Length: 81, dtype: int64
count      1460.000000
mean     180921.195890
std       79442.502883
min       34900.000000
25%      129975.000000
50% 

In [8]:
from sklearn.model_selection import cross_validate   #交叉验证
from sklearn.metrics import make_scorer, mean_squared_error
def rmse_original_scale(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return np.sqrt(mean_squared_error(y_true, y_pred))

scoring = {
    'r2': 'r2',
    'rmse': make_scorer(rmse_original_scale, greater_is_better=False)
}

results = cross_validate(
    xgb.XGBRegressor(
        n_estimators=452, learning_rate=0.05, max_depth=3,
        min_child_weight=5, reg_alpha=0.5, reg_lambda=2.0,
        subsample=0.7, colsample_bytree=0.6, random_state=42
    ),
    X_final, y_log, cv=5, scoring=scoring
)

r2_scores = results['test_r2']
rmse_scores = -results['test_rmse']

print(f'5折 R²:   {r2_scores.mean():.4f} ± {r2_scores.std():.4f}')
print(f'5折 RMSE: {rmse_scores.mean():.2f} ± {rmse_scores.std():.2f} 美元')

5折 R²:   0.9041 ± 0.0078
5折 RMSE: 25698.88 ± 4014.96 美元
